In [1]:
# Primeira Liga Top-5 (2017/18 … 2024/25) — robust link scraper from https://www.football-data.co.uk/portugalm.php

import io, re, time, requests, pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin

PAGE_URL = "https://www.football-data.co.uk/portugalm.php"
UA = {"User-Agent": "Mozilla/5.0 (compatible; PrimeiraLigaTop5/1.0)"} 
# User-Agent tells the website what client is making the request. Many sites block “unknown” clients

OUT_DIR = Path("data").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# season codes shown on that page
TARGET_CODES = [f"{y%100:02d}{(y+1)%100:02d}" for y in range(2017, 2025)]  # 1718..2425

def season_label_from_code(code: str) -> str:
    y0 = 2000 + int(code[:2])
    return f"{y0}/{str(y0+1)[-2:]}"

def fetch_page_links() -> dict:
    """Return {code: absolute_url_to_P1.csv} from portugalm.php.
       Handles relative/absolute hrefs; accepts trailing querystrings."""
    r = requests.get(PAGE_URL, headers=UA, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    links = {}
    pat = re.compile(r"mmz4281/(\d{4})/P1\.csv(?:\?.*)?$", re.I)
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        m = pat.search(href)
        if not m:
            continue
        code = m.group(1)
        abs_url = urljoin(PAGE_URL, href)   # fixes relative hrefs like 'mmz4281/2425/P1.csv'
        links[code] = abs_url

    # Fallback: construct links directly if nothing matched (just in case)
    if not links:
        base = "https://www.football-data.co.uk/mmz4281/{code}/P1.csv"
        links = {code: base.format(code=code) for code in TARGET_CODES}
    return links

def download_csvs(codes_to_urls: dict) -> dict:
    saved = {}
    for code in TARGET_CODES:
        if code not in codes_to_urls:
            continue
        url = codes_to_urls[code]
        fname = OUT_DIR / f"P1_{code}_{season_label_from_code(code).replace('/', '-')}.csv"
        if not fname.exists():
            r = requests.get(url, headers=UA, timeout=60)
            r.raise_for_status()
            fname.write_bytes(r.content)
            time.sleep(1.0)
        saved[code] = str(fname)
    return saved

In [2]:
def table_from_matches(df: pd.DataFrame) -> pd.DataFrame:
    """Compute full league table (3 pts/win) from Football-Data match CSV."""
    req = {"HomeTeam", "AwayTeam", "FTHG", "FTAG"}
    if not req.issubset(df.columns):
        missing = req - set(df.columns)
        raise ValueError(f"CSV missing columns: {missing}")

    # Work on a copy and create helper columns
    m = df[["HomeTeam", "AwayTeam", "FTHG", "FTAG"]].copy()
    m["HW"] = (m["FTHG"] > m["FTAG"]).astype(int)    # home wins
    m["D"]  = (m["FTHG"] == m["FTAG"]).astype(int)   # draws
    m["AW"] = (m["FTHG"] < m["FTAG"]).astype(int)    # away wins

    # Home aggregates
    home = (
        m.groupby("HomeTeam")
         .agg(
             PldH=("HomeTeam", "size"),
             WH=("HW", "sum"),
             DH=("D", "sum"),
             LH=("AW", "sum"),
             GFH=("FTHG", "sum"),
             GAH=("FTAG", "sum"),
         )
         .rename_axis("Team")
    )

    # Away aggregates
    away = (
        m.groupby("AwayTeam")
         .agg(
             PldA=("AwayTeam", "size"),
             WA=("AW", "sum"),      # away-team wins
             DA=("D", "sum"),
             LA=("HW", "sum"),      # away-team losses = home wins
             GFA=("FTAG", "sum"),
             GAA=("FTHG", "sum"),
         )
         .rename_axis("Team")
    )

    # Combine
    t = home.join(away, how="outer").fillna(0).astype(int)
    t["Matches"] = t["PldH"] + t["PldA"]
    t["W"]   = t["WH"] + t["WA"]
    t["D"]   = t["DH"] + t["DA"]
    t["L"]   = t["LH"] + t["LA"]
    t["Goals for"]  = t["GFH"] + t["GFA"]
    t["Goals against"]  = t["GAH"] + t["GAA"]
    t["Goal difference"]  = t["Goals for"] - t["Goals against"]
    t["Points"] = t["W"]*3 + t["D"]

    t = t[["Matches","W","D","L","Goals for","Goals against","Goal difference","Points"]].sort_values(
        ["Points", "Goal difference", "Goals for", "Team"], ascending=[False, False, False, True]
    )
    t["Place"] = range(1, len(t) + 1)
    return t.reset_index()

def top5_from_file(path: str, season_label: str) -> pd.DataFrame:
    # Football-Data often uses Windows-1252/latin-1
    df = pd.read_csv(path, encoding="latin-1")
    tab = table_from_matches(df)
    top5 = tab.loc[tab["Place"] <= 5, ["Matches","Place","Team","Points","Goal difference","Goals for","Goals against"]].copy()
    top5.insert(0, "Season", season_label)
    return top5

In [3]:
links = fetch_page_links()
print(f"Found season CSV links on page: {sorted(links.keys())}")

Found season CSV links on page: ['0001', '0102', '0203', '0304', '0405', '0506', '0607', '0708', '0809', '0910', '1011', '1112', '1213', '1314', '1415', '1516', '1617', '1718', '1819', '1920', '2021', '2122', '2223', '2324', '2425', '2526', '9495', '9596', '9697', '9798', '9899', '9900']


In [4]:
# we will save cvs-files with results of all matches from seasons 17/18 to 24/25 to our data folder
saved = download_csvs(links)

In [5]:
# printing top-5 places from seasons 2017/18 to 2024/25 and saving the same info in CSV-file to our data folder
tops = [top5_from_file(path, season_label_from_code(code)) for code, path in sorted(saved.items())]
top5_all = pd.concat(tops, ignore_index=True)

from IPython.display import display
display(top5_all)

out = OUT_DIR / "primeira_liga_top5_1718_2425.csv"
top5_all.to_csv(out, index=False, encoding="utf-8-sig")

,Season,Matches,Place,Team,Points,Goal difference,Goals for,Goals against
0,2017/18,34,1,Porto,88,64,82,18
1,2017/18,34,2,Benfica,81,58,80,22
2,2017/18,34,3,Sp Lisbon,78,39,63,24
3,2017/18,34,4,Sp Braga,75,45,74,29
4,2017/18,34,5,Rio Ave,51,-2,40,42
5,2018/19,34,1,Benfica,87,72,103,31
6,2018/19,34,2,Porto,85,54,74,20
7,2018/19,34,3,Sp Lisbon,74,39,72,33
8,2018/19,34,4,Sp Braga,67,19,56,37
9,2018/19,34,5,Guimaraes,52,12,46,34


In [6]:
import re
import pandas as pd
from pathlib import Path

# --- paths ---
DATA_DIR = Path("data")
FILES = sorted(DATA_DIR.glob("P1_*.csv"))

# keep seasons 2017/18 … 2024/25 => codes 1718..2425 in filenames
KEEP = {f"{y%100:02d}{(y+1)%100:02d}" for y in range(2017, 2025)}
FILES = [p for p in FILES if (m:=re.search(r"P1_(\d{4})_", p.name)) and m.group(1) in KEEP]
if not FILES:
    raise FileNotFoundError(f"No season files found in {DATA_DIR} matching P1_*.csv for 2017/18–2024/25")

def season_label_from_fname(fn: str) -> str:
    m = re.search(r"(\d{4}-\d{2})", fn)
    if m:
        y0 = int(m.group(1)[:4]); return f"{y0}/{str(y0+1)[-2:]}"
    m = re.search(r"P1_(\d{2})(\d{2})_", fn)
    if m:
        y0 = 2000 + int(m.group(1)); return f"{y0}/{str(y0+1)[-2:]}"
    return "Unknown"

def norm_team(name: str) -> str:
    n = str(name).strip().lower()
    if (n.startswith("sporting") or n.startswith("sp")) and "braga" not in n:
        return "Sporting"
    if "benfica" in n:
        return "Benfica"
    return str(name)

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    if "FTHG" not in df.columns and "HG" in df.columns:
        df = df.rename(columns={"HG": "FTHG"})
    if "FTAG" not in df.columns and "AG" in df.columns:
        df = df.rename(columns={"AG": "FTAG"})
    return df

def build_table(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_columns(df)
    req = {"HomeTeam","AwayTeam","FTHG","FTAG"}
    if missing := (req - set(df.columns)):
        raise ValueError(f"Missing columns for table build: {missing}")

    m = df[["HomeTeam","AwayTeam","FTHG","FTAG"]].copy()
    m["HW"] = (m["FTHG"] > m["FTAG"]).astype(int)
    m["D"]  = (m["FTHG"] == m["FTAG"]).astype(int)
    m["AW"] = (m["FTHG"] < m["FTAG"]).astype(int)

    home = (m.groupby("HomeTeam")
              .agg(PldH=("HomeTeam","size"), WH=("HW","sum"), DH=("D","sum"),
                   LH=("AW","sum"), GFH=("FTHG","sum"), GAH=("FTAG","sum"))
              .rename_axis("Team"))
    away = (m.groupby("AwayTeam")
              .agg(PldA=("AwayTeam","size"), WA=("AW","sum"), DA=("D","sum"),
                   LA=("HW","sum"), GFA=("FTAG","sum"), GAA=("FTHG","sum"))
              .rename_axis("Team"))
    t = home.join(away, how="outer").fillna(0).astype(int)
    t["Pld"] = t["PldH"] + t["PldA"]
    t["W"]   = t["WH"] + t["WA"]
    t["D"]   = t["DH"] + t["DA"]
    t["L"]   = t["LH"] + t["LA"]
    t["GF"]  = t["GFH"] + t["GFA"]
    t["GA"]  = t["GAH"] + t["GAA"]
    t["GD"]  = t["GF"] - t["GA"]
    t["Pts"] = t["W"]*3 + t["D"]
    t = t[["Pld","W","D","L","GF","GA","GD","Pts"]].sort_values(
        ["Pts","GD","GF","Team"], ascending=[False, False, False, True]
    ).reset_index()
    t["NormTeam"] = t["Team"].apply(norm_team)
    t["Pos"] = range(1, len(t)+1)
    return t

def per_team_stats(df: pd.DataFrame, team: str) -> dict:
    df = normalize_columns(df)
    for c in ("HST","AST"):
        if c not in df.columns: df[c] = pd.NA

    req = {"HomeTeam","AwayTeam","FTHG","FTAG"}
    if missing := (req - set(df.columns)):
        raise ValueError(f"Missing columns after normalization: {missing}")

    home = df[df["HomeTeam"].apply(norm_team) == team].copy()
    away = df[df["AwayTeam"].apply(norm_team) == team].copy()
    hmatches, amatches = len(home), len(away)
    matches = hmatches + amatches

    gf = int(home["FTHG"].sum() + away["FTAG"].sum())
    ga = int(home["FTAG"].sum() + away["FTHG"].sum())
    gd = gf - ga

    wins_home = int((home["FTHG"] > home["FTAG"]).sum())
    wins_away = int((away["FTAG"] > away["FTHG"]).sum())
    wins = wins_home + wins_away
    draws = int((home["FTHG"] == home["FTAG"]).sum() + (away["FTAG"] == away["FTHG"]).sum())
    points = wins*3 + draws

    win_pct = (wins / matches * 100.0) if matches else float("nan")
    home_win_pct = (wins_home / hmatches * 100.0) if hmatches else float("nan")

    if matches:
        tot_goals_series = pd.concat(
            [(home["FTHG"] + home["FTAG"]), (away["FTHG"] + away["FTAG"])],
            ignore_index=True
        )
        avg_goals_both = float(tot_goals_series.mean())
    else:
        avg_goals_both = float("nan")

    HST_h = pd.to_numeric(home.get("HST"), errors="coerce")
    AST_h = pd.to_numeric(home.get("AST"), errors="coerce")
    HST_a = pd.to_numeric(away.get("HST"), errors="coerce")
    AST_a = pd.to_numeric(away.get("AST"), errors="coerce")

    sot_for_season = float(HST_h.sum(skipna=True) + AST_a.sum(skipna=True))
    sot_against_season = float(AST_h.sum(skipna=True) + HST_a.sum(skipna=True))

    tot_sot_home = (HST_h + AST_h).dropna()
    tot_sot_away = (HST_a + AST_a).dropna()
    tot_sot = pd.concat([tot_sot_home, tot_sot_away], ignore_index=True)
    avg_sot_both = float(tot_sot.mean()) if not tot_sot.empty else float("nan")

    return {
        "Matches": matches,
        "Points": points,
        "WinPct": win_pct,            # keep numeric; format later
        "HomeWinPct": home_win_pct,   # keep numeric; format later
        "Goals for": gf, "Goals against": ga, "Goal difference": gd, 
        "Shots_on_target_for": int(round(sot_for_season)) if pd.notna(sot_for_season) else None,
        "Shots_on_target_against": int(round(sot_against_season)) if pd.notna(sot_against_season) else None,
        "Avg_goals_in_match": round(avg_goals_both, 3) if pd.notna(avg_goals_both) else None,
        "Avg_shots_on_target_in_match": round(avg_sot_both, 3) if pd.notna(avg_sot_both) else None,
    }

def fmt_pct(x):
    return f"{int(round(x))}%" if pd.notna(x) else None

In [7]:
# ---- PER-SEASON TABLE ----
rows_season = []
for p in FILES:
    try:
        df = pd.read_csv(p, encoding="latin-1")
    except Exception:
        df = pd.read_csv(p, encoding="utf-8")

    try:
        season = season_label_from_fname(p.name)
        table = build_table(df)
        for club in ["Sporting", "Benfica"]:
            s = per_team_stats(df, club)
            pos = table.loc[table["NormTeam"] == club, "Pos"]
            place = int(pos.iloc[0]) if not pos.empty else None
            rows_season.append({
                "Season": season, "Team": club,
                "Matches": s["Matches"], "Place": place,
                "Points": s["Points"],
                "WinPct": s["WinPct"], "HomeWinPct": s["HomeWinPct"],
                "Goals for": s["Goals for"], "Goals against": s["Goals against"], "Goal difference": s["Goal difference"],
                "Shots_on_target_for": s["Shots_on_target_for"], "Shots_on_target_against": s["Shots_on_target_against"],
                "Avg_goals_in_match": s["Avg_goals_in_match"], "Avg_shots_on_target_in_match": s["Avg_shots_on_target_in_match"],
            })
    except Exception as e:
        print(f"Skipping {p.name}: {e}")

if not rows_season:
    raise RuntimeError("No season rows were built — check that your P1_* files exist and contain FTHG/FTAG or HG/AG.")

season_df = pd.DataFrame(rows_season)
# format % columns
season_df["WinPct"] = season_df["WinPct"].apply(fmt_pct)
season_df["HomeWinPct"] = season_df["HomeWinPct"].apply(fmt_pct)

# column order
cols_order = ["Season","Team","Matches","Place","Points","WinPct","HomeWinPct",
              "Goal difference","Goals for","Goals against","Shots_on_target_for","Shots_on_target_against",
              "Avg_goals_in_match","Avg_shots_on_target_in_match"]
season_df = season_df[cols_order].sort_values(["Season","Team"]).reset_index(drop=True)
display(season_df)
season_df.to_csv(DATA_DIR / "sporting_benfica_totals_per_season_1718_2425.csv",
                 index=False, encoding="utf-8-sig")
print(f"Saved: {DATA_DIR / 'sporting_benfica_totals_per_season_1718_2425.csv'}")

,Season,Team,Matches,Place,Points,WinPct,HomeWinPct,Goal difference,Goals for,Goals against,Shots_on_target_for,Shots_on_target_against,Avg_goals_in_match,Avg_shots_on_target_in_match
0,2017/18,Benfica,34,2,81,74%,82%,58,80,22,237,84,3.000,9.441
1,2017/18,Sporting,34,3,78,71%,82%,39,63,24,176,107,2.559,8.324
2,2018/19,Benfica,34,1,87,82%,82%,72,103,31,227,118,3.941,10.147
3,2018/19,Sporting,34,3,74,68%,82%,39,72,33,199,122,3.088,9.441
4,2019/20,Benfica,34,2,77,71%,71%,45,71,26,189,106,2.853,8.676
5,2019/20,Sporting,34,4,60,53%,71%,15,49,34,144,118,2.441,7.706
6,2020/21,Benfica,34,3,76,68%,71%,42,69,27,174,116,2.824,8.529
7,2020/21,Sporting,34,1,85,76%,76%,45,65,20,163,77,2.500,7.059
8,2021/22,Benfica,34,3,74,68%,59%,48,78,30,200,112,3.176,9.176
9,2021/22,Sporting,34,2,85,79%,82%,50,73,23,209,83,2.824,8.588


Saved: data\sporting_benfica_totals_per_season_1718_2425.csv


In [8]:
# ---- ALL-SEASONS (NO SEASON SPLIT) ----
dflist = []
for p in FILES:
    try:
        dflist.append(pd.read_csv(p, encoding="latin-1"))
    except Exception:
        dflist.append(pd.read_csv(p, encoding="utf-8"))
big = pd.concat(dflist, ignore_index=True)

rows_all = []
for club in ["Sporting","Benfica"]:
    s = per_team_stats(big, club)
    rows_all.append({
        "Team": club,
        "Matches": s["Matches"], "Points": s["Points"],
        "WinPct": s["WinPct"], "HomeWinPct": s["HomeWinPct"],
        "Goals for": s["Goals for"], "Goals against": s["Goals against"], "Goal difference": s["Goal difference"],
        "Shots_on_target_for": s["Shots_on_target_for"], "Shots_on_target_against": s["Shots_on_target_against"],
        "Avg_goals_in_match": s["Avg_goals_in_match"], "Avg_shots_on_target_in_match": s["Avg_shots_on_target_in_match"],
    })

all_df = pd.DataFrame(rows_all)
all_df["WinPct"] = all_df["WinPct"].apply(fmt_pct)
all_df["HomeWinPct"] = all_df["HomeWinPct"].apply(fmt_pct)

cols_all = ["Team","Matches","Points","WinPct","HomeWinPct",
            "Goal difference","Goals for","Goals against","Shots_on_target_for","Shots_on_target_against",
            "Avg_goals_in_match","Avg_shots_on_target_in_match"]
all_df = all_df[cols_all].sort_values("Team").reset_index(drop=True)
display(all_df)
all_df.to_csv(DATA_DIR / "sporting_benfica_totals_in_1718_2425.csv.csv",
              index=False, encoding="utf-8-sig")
print(f"Saved: {DATA_DIR / 'sporting_benfica_totals_in_1718_2425.csv'}")

,Team,Matches,Points,WinPct,HomeWinPct,Goal difference,Goals for,Goals against,Shots_on_target_for,Shots_on_target_against,Avg_goals_in_match,Avg_shots_on_target_in_match
0,Benfica,272,642,74%,78%,432,644,212,1695,837,3.147,9.309
1,Sporting,272,628,72%,82%,355,577,222,1530,756,2.938,8.404


Saved: data\sporting_benfica_totals_in_1718_2425.csv
